# OpenCV & NumPy 학습 정리

---

## 1. np.uint8 오타 오류

**Q. `np.unit8` 오류가 떴어요.**

`unit8`이 아니라 `uint8` (unsigned integer 8-bit)이 올바른 표기입니다.

```python
np.full((100, 256), 255, dtype=np.uint8)
```

> `np.uint8`은 0~255 범위만 저장 가능하므로 256을 넣으면 오버플로우로 0이 됩니다.

---

## 2. Otsu 알고리즘

**Q. Otsu가 어떤 역할을 하는 건가요?**

이미지를 흑/백으로 나눌 때 **최적의 임계값(threshold)을 자동으로 찾아주는 알고리즘**입니다.

- 픽셀을 두 그룹(어두운/밝은)으로 나눌 때 **그룹 간 분산은 최대화**, **그룹 내 분산은 최소화**하는 지점을 임계값으로 선택
- `cv2.THRESH_BINARY + cv2.THRESH_OTSU`로 사용

```python
result, binary_image_otsu = cv2.threshold(src_gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
# result = 149.0 → Otsu가 찾은 최적 임계값
```

---

## 3. 비트 플래그 (cv2.THRESH_BINARY + cv2.THRESH_OTSU)

**Q. 왜 `+`를 쓰고 어떻게 되는 건가요?**

여기서 `+`는 **두 옵션을 동시에 적용**하는 비트 플래그 방식입니다.

```
cv2.THRESH_BINARY = 0  (0000 0000)
cv2.THRESH_OTSU   = 8  (0000 1000)
0 + 8 = 8
```

- 각 옵션이 서로 다른 비트 자리를 차지해 더해도 충돌 없음
- `|` (OR 연산자)가 더 정확한 표현이지만 `+`도 동일하게 동작

**실행 순서:**
1. THRESH_OTSU → 최적 임계값 자동 계산
2. THRESH_BINARY → 그 임계값으로 흑/백 이진화

---

## 4. 신호 디지털화: Sampling & Quantization & Encoding

### Sampling (샘플링)
**시간 축**을 일정 간격으로 끊어서 값을 추출하는 것

- 단위: Hz (1초에 몇 번 측정)
- CD 음질 = 44,100 Hz

### Quantization (양자화)
**값 축**을 정해진 단계로 반올림하는 것

- 8bit → 0~255 (256단계)
- 16bit → 0~65535 (65536단계)

### Encoding (인코딩)
Sampling → Quantization으로 얻은 숫자를 **0과 1(비트)로 변환**하는 마지막 단계

```
149 → 1001 0101  (8bit)
255 → 1111 1111
  0 → 0000 0000
```

| | Sampling | Quantization |
|---|---|---|
| 의미 | 픽셀 수 | 색상 단계 |
| 단위 | px (해상도) | bit (색 깊이) |

---

## 5. Sampling Rate & Nyquist Rate

### Sampling Rate
1초에 몇 번 샘플링하는가를 나타내는 값 (단위: Hz)

### Nyquist Rate
원본 신호를 완벽하게 복원하기 위한 **최소 샘플링 레이트**

```
Nyquist Rate = 원본 신호의 최대 주파수 × 2
```

- 신호의 한 주기를 표현하려면 최소 2개의 샘플이 필요하기 때문
- Sampling Rate < Nyquist Rate → **앨리어싱(Aliasing)** 발생

---

## 6. HSV 색공간

**Q. HSV가 딥러닝에서 색 인식을 잘 하나요?**

| 요소 | 의미 | 범위 |
|---|---|---|
| H (Hue) | 색의 종류 | 0° ~ 360° |
| S (Saturation) | 색의 선명함 | 0% ~ 100% |
| V (Value) | 색의 밝기 | 0% ~ 100% |

- **조명 변화에 강함**: 조명이 바뀌면 V(밝기)만 변하고 H(색상)는 유지
- RGB는 조명이 바뀌면 R, G, B 세 값이 모두 변함
- 딥러닝에서는 RGB로도 충분 (모델이 알아서 학습)

| 상황 | 추천 |
|---|---|
| 특정 색 범위 추출 (마스킹) | HSV |
| 조명 변화가 심한 환경 | HSV |
| 일반적인 딥러닝 학습 | RGB |

---

## 7. uint8

**부호 없는 8비트 정수** (Unsigned Integer 8-bit)

```
u    → unsigned (부호 없음, 음수 없음)
int  → integer (정수)
8    → 8비트 → 2⁸ = 256가지 → 0 ~ 255
```

**오버플로우 주의:**
```python
np.uint8(255) + 1 = 0    # 255 넘으면 다시 0으로
np.uint8(0) - 1 = 255    # 0 아래로 내려가면 255로
```

---

## 8. np.clip을 사용한 밝기 조절

```python
np.clip(src.astype(np.uint16) + 50, 0, 255).astype(np.uint8)
```

| 단계 | 코드 | 이유 |
|---|---|---|
| ① | `src.astype(np.uint16)` | 오버플로우 방지용 형변환 |
| ② | `+ 50` | 밝기 증가 |
| ③ | `np.clip(..., 0, 255)` | 255 초과값을 255로 고정 |
| ④ | `.astype(np.uint8)` | 이미지 형식으로 되돌림 |

- uint16 → uint8 변환 시 자동으로 값을 낮춰주지 않고 비트를 잘라버림
- 예: `270 = 1 0000 1110` → uint8로 변환 시 앞 비트 잘림 → `14`

---

## 9. 비트 연산 (AND / OR / XOR)

```
AND → 둘 다 1일 때만 1  (교집합)
OR  → 하나라도 1이면 1  (합집합)
XOR → 서로 다를 때만 1  (차집합 느낌)
```

모두 **픽셀 숫자 단위가 아니라 비트 단위**로 연산합니다.

```
125 = 0111 1101
 40 = 0010 1000

OR  = 0111 1101 = 125
XOR = 0101 0101 = 85
```

---

## 10. plt.subplots

```python
fig, axes = plt.subplots(1, 2, figsize=(6, 3))
```

- `1, 2` → 1행 2열 그래프 창
- `figsize=(6, 3)` → 가로 6인치, 세로 3인치
- `axes[0]`, `axes[1]`로 각 칸에 이미지 삽입

---

## 11. plt.imshow의 컬러맵 자동 적용

2차원(흑백) 이미지를 `plt.imshow()`로 표시하면 **자동으로 컬러맵(viridis) 적용**됩니다.

```python
axes[0].imshow(img, cmap='gray')  # 흑백으로 표시하려면 cmap='gray' 명시
```

---